# Mamba-SSM Oracle — single diagnostic comparison

**Project: `transformer-vs-ssm` (sequence-modeling blog series). Unrelated to any competition.**

## Why this exists

After fixing three real bugs in the from-scratch Mamba (`dt_proj` init, a blanket
reinit that erased it, and the missing `dt_rank` bottleneck), plus proving scan
parity against the official reference (`tests/test_selective_scan_parity.py`,
passes), the corrected model **still fails Selective Copying** at `ssm_lr_scale=1`
(commit `58ebc92`) -- flat loss at ln(64) through 800 steps, val_acc 0.008-0.012
vs. chance 0.032.

Per external review: when scan parity passes and the model still fails, the next
discriminating experiment is the **official `mamba-ssm` package as an oracle** --
same task, same dimensions, same optimizer/schedule -- to find out whether the
remaining gap is (a) still something specific to our implementation, or (b) the
task/training recipe itself at this scale/step budget.

## Decision table

| Custom Mamba | Official `mamba-ssm` | Interpretation |
|---|---|---|
| fails | **learns** | model-level discrepancy remains (wrapper/precision/training path) |
| fails | **fails** | stop debugging the architecture -- investigate task/recipe first |
| learns | learns | custom implementation validated |
| learns | fails | harness/config mismatch -- don't interpret either yet |

This notebook trains **three models, one seed each, same task/steps**, for a clean
comparison in one environment: Transformer (known-good control), our custom Mamba
(post-fix), official `mamba_ssm.modules.mamba_simple.Mamba` (Mamba-1, matching
what we've been debugging -- not Mamba-2).

## Kaggle settings
- Accelerator: **GPU T4 x1**, Internet **ON**
- Save Version -> Run All
- If `mamba-ssm` fails to install/import, the notebook stops with a clear
  diagnostic rather than failing opaquely later.

In [ ]:
# Cell 1: Clone (or refresh) the repo -- picks up the ablation fix (7ec4346)
import os, subprocess, sys, platform

REPO = "https://github.com/nvaidyan1/transformer-vs-ssm.git"
if not os.path.exists("transformer-vs-ssm"):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir("/kaggle/working/transformer-vs-ssm")
subprocess.run(["git", "pull", "--ff-only"], check=False)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

import torch
print(f"\npython {platform.python_version()}  torch {torch.__version__}  "
      f"cuda {torch.version.cuda}  "
      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

In [ ]:
# Cell 2: Repo dependencies + full test suite (must pass before spending GPU time)
!pip install -q pyyaml pytest einops 2>&1 | tail -1
!python -m pytest tests/ -q 2>&1 | tail -6

In [ ]:
# Cell 3: Install mamba-ssm. causal-conv1d is optional (best-effort acceleration
# for the depthwise conv; the module falls back to plain PyTorch conv without it).
# mamba-ssm itself is NOT optional -- its compiled selective_scan extension is
# imported unconditionally by mamba_simple.py, so a failed build here blocks the
# whole notebook.
#
# Attempt 1 (kernel v1) failed at "Getting requirements to build wheel" -- pip's
# outer error, with the real cause hidden because `-q` suppressed the subprocess's
# own output. Root cause (well-documented for this package): mamba-ssm's setup.py
# needs to introspect the ALREADY-INSTALLED torch/CUDA version to pick the right
# build path, but pip's isolated build sandbox (PEP 517) doesn't have torch
# visible by default. Fix: --no-build-isolation, plus pre-installing the build-time
# deps setup.py itself needs (packaging, ninja, wheel, psutil). No -q this time --
# if it fails again, the full output below must be enough to diagnose it further,
# not just "exit code 1" again.
import subprocess

def run(cmd):
    print(f"$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
    print(f"[exit {r.returncode}]\n")
    return r

run(["pip", "install", "packaging", "ninja", "wheel", "psutil"])

print("Attempting causal-conv1d (optional, --no-build-isolation)...")
r = run(["pip", "install", "causal-conv1d>=1.4.0", "--no-build-isolation"])
print("  OK" if r.returncode == 0 else "  not available (non-fatal) -- see output above")

print("\nAttempting mamba-ssm (required, --no-build-isolation)...")
r = run(["pip", "install", "mamba-ssm", "--no-build-isolation"])
if r.returncode != 0:
    print("=" * 70)
    print("mamba-ssm FAILED TO INSTALL even with --no-build-isolation. Full output")
    print("is above (not truncated this time). The oracle experiment cannot")
    print("proceed on this environment -- stop here rather than fail later.")
    print("=" * 70)
    raise SystemExit("mamba-ssm install failed -- see diagnostic above")
print("  OK")

try:
    from mamba_ssm.modules.mamba_simple import Mamba as OfficialMamba
    import mamba_ssm
    print(f"\nmamba_ssm {mamba_ssm.__version__} imported OK")
except Exception as e:
    print("=" * 70)
    print(f"mamba-ssm installed but failed to IMPORT: {type(e).__name__}: {e}")
    print("Likely a torch/cuda ABI mismatch between the installed wheel and this")
    print("environment. Stop here -- see the version print in Cell 1.")
    print("=" * 70)
    raise

In [ ]:
# Cell 4: Oracle wrapper -- residual pre-norm block around the official Mamba-1
# mixer, matching our own MambaBlock structure (x + mixer(norm(x))) and our
# model's forward(x) -> logits interface, so train/eval code can be shared.
#
# use_fast_path is exposed and passed straight through to the official Mamba
# constructor. This is the discriminator recommended after the first oracle
# result: with use_fast_path=True it calls the fused mamba_inner_fn; with it
# False, it separately runs conv1d, x_proj, dt_proj, selective_scan_fn, then
# out_proj -- same parameters and architecture, different execution path.
# Note use_fast_path=False can still use optimized causal_conv1d_fn/
# selective_scan_fn CUDA ops if installed -- it is not "pure PyTorch", so if
# it also learns well, the next narrowing step is module-level weight-copy
# parity, not "CUDA vs Python".
import torch
import torch.nn as nn

class OracleBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand, layer_idx, use_fast_path=True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mixer = OfficialMamba(d_model=d_model, d_state=d_state, d_conv=d_conv,
                                   expand=expand, layer_idx=layer_idx,
                                   use_fast_path=use_fast_path)

    def forward(self, x):
        return x + self.mixer(self.norm(x))

class OracleMamba(nn.Module):
    def __init__(self, vocab_size, n_layers, d_model, d_state, d_conv, expand,
                use_fast_path=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            OracleBlock(d_model, d_state, d_conv, expand, layer_idx=i,
                       use_fast_path=use_fast_path)
            for i in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying, matching our model

    def forward(self, x):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# Cell 5: Shared settings -- identical across all three models
TASK_ARGS = dict(n_data=4, seq_len=64, vocab_size=64)
D_MODEL, D_STATE, D_CONV, EXPAND, N_LAYERS = 124, 16, 4, 2, 4  # matches mamba_small.yaml
BATCH, LR, WARMUP, STEPS, SEED = 32, 3e-4, 200, 2000, 0
OUT = "/kaggle/working/oracle"

import os
os.makedirs(OUT, exist_ok=True)
print(f"task={TASK_ARGS}  d_model={D_MODEL} d_state={D_STATE} d_conv={D_CONV} "
      f"expand={EXPAND} n_layers={N_LAYERS}\n"
      f"batch={BATCH} lr={LR} warmup={WARMUP} steps={STEPS} seed={SEED}")

In [ ]:
# Cell 6: Transformer -- known-good control, via the existing harness (unchanged)
task_args_str = ",".join(f"{k}={v}" for k, v in TASK_ARGS.items())
!python src/train_task.py --config configs/recall/transformer_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --seed {SEED} --device cuda \
    --out {OUT}/transformer.json 2>&1 | tail -6

In [ ]:
# Cell 7: Our custom Mamba, post-fix, ssm_lr_scale=1 (default -- no boost)
!python src/train_task.py --config configs/recall/mamba_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --lr {LR} --seed {SEED} --device cuda \
    --out {OUT}/custom_mamba.json 2>&1 | tail -6

In [ ]:
# Cell 8: Official mamba-ssm oracle, trained TWICE -- use_fast_path=True and
# =False -- same seed, same everything else. Discriminator recommended after
# the first oracle result: if both learn similarly, fusion is exonerated and
# the gap is at the block/model level, not CUDA-fused-vs-naive execution. If
# only the fast path learns, the execution path itself is implicated.
import sys, math, time, json
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
import torch
import torch.nn as nn
from src.tasks import IGNORE_INDEX, make_batcher

def train_oracle(use_fast_path, out_path, seed=SEED, steps=STEPS):
    torch.manual_seed(seed)
    oracle = OracleMamba(vocab_size=TASK_ARGS["vocab_size"], n_layers=N_LAYERS,
                         d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                         expand=EXPAND, use_fast_path=use_fast_path).to(device)
    n_params = count_params(oracle)

    decay, no_decay = [], []
    for n, p in oracle.named_parameters():
        (no_decay if getattr(p, "_no_weight_decay", False) else decay).append(p)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
        lr=LR, betas=(0.9, 0.95))

    def get_lr(step, warmup=WARMUP, max_steps=steps, lr=LR):
        if step < warmup:
            return lr * (step + 1) / warmup
        prog = (step - warmup) / max(1, max_steps - warmup)
        return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))

    batcher = make_batcher("selective_copy", **TASK_ARGS)
    t0 = time.time()
    oracle.train()
    history = []
    for step in range(steps):
        lr = get_lr(step)
        for g in opt.param_groups:
            g["lr"] = lr
        tok, tgt, _, _ = batcher(BATCH, seed=step + seed * 10_000_000)
        tok = torch.from_numpy(tok).to(device)
        tgt = torch.from_numpy(tgt).to(device)
        logits = oracle(tok)
        loss = nn.functional.cross_entropy(logits.reshape(-1, TASK_ARGS["vocab_size"]),
                                           tgt.reshape(-1), ignore_index=IGNORE_INDEX)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(oracle.parameters(), 1.0)
        opt.step()

        if (step + 1) % 250 == 0 or step == steps - 1:
            oracle.eval()
            with torch.no_grad():
                vlogits = oracle(val_tok)
                vloss = nn.functional.cross_entropy(
                    vlogits.reshape(-1, TASK_ARGS["vocab_size"]), val_tgt.reshape(-1),
                    ignore_index=IGNORE_INDEX)
                vacc = (vlogits.argmax(-1)[val_mask] == val_tgt[val_mask]).float().mean().item()
            oracle.train()
            history.append({"step": step + 1, "val_acc": vacc, "val_loss": vloss.item()})
            print(f"  [{'fast' if use_fast_path else 'slow'}] step {step+1:>5d}  "
                  f"train_loss {loss.item():.4f}  val_loss {vloss.item():.4f}  "
                  f"val_acc {vacc:.4f}  ({time.time()-t0:.0f}s)", flush=True)

    record = {"arch": f"mamba_ssm_oracle_{'fast' if use_fast_path else 'slow'}",
             "use_fast_path": use_fast_path, "n_params": n_params,
             "final_val_acc": vacc, "final_val_loss": vloss.item(),
             "steps_run": steps, "seed": seed,
             "wall_clock_s": round(time.time() - t0, 1),
             "chance_acc": 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2),
             "history": history}
    json.dump(record, open(out_path, "w"), indent=2)
    print(f"  final val_acc {vacc:.4f}  ({record['wall_clock_s']}s)\n")
    return record

device = torch.device("cuda")
val_tok, val_tgt, val_mask, _ = make_batcher("selective_copy", **TASK_ARGS)(64, seed=10_000_000)
val_tok, val_tgt, val_mask = (torch.from_numpy(val_tok).to(device),
                              torch.from_numpy(val_tgt).to(device),
                              torch.from_numpy(val_mask).to(device))

print("=== Official mamba-ssm, use_fast_path=True (fused) ===")
rec_fast = train_oracle(True, f"{OUT}/oracle_fast.json")

print("=== Official mamba-ssm, use_fast_path=False (unfused, still may use CUDA ops) ===")
rec_slow = train_oracle(False, f"{OUT}/oracle_slow.json")

In [ ]:
# Cell 9: Verdict -- trajectory-aware, not a single endpoint threshold.
#
# A hard endpoint cutoff (e.g. "val_acc > 0.5") mis-classifies a model that is
# clearly still improving at the step budget's end as a "fail", which is what
# happened in kernel v2 (oracle 0.46 and rising steeply was called the same as
# custom 0.13). The categories below look at both level and trend:
#
#   LEARNS       clearly above chance AND still improving meaningfully over the
#                second half of training (or already high and stable)
#   FLAT/FAIL    stays near chance, no meaningful improvement over training
#   INCONCLUSIVE improving, but not clearly separated from chance/the other run
import glob, json

results = {}
for f in glob.glob(f"{OUT}/*.json"):
    r = json.load(open(f))
    key = r.get("arch", r.get("task", "unknown"))
    results[key] = r

chance = 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2)

def classify(rec, chance):
    if rec is None:
        return "N/A", None
    acc = rec["final_val_acc"]
    hist = rec.get("history", [])
    if len(hist) >= 2:
        mid = hist[len(hist) // 2]["val_acc"]
        still_rising = (acc - mid) > 0.03  # meaningful improvement in the second half
    else:
        still_rising = False
    if acc < chance * 2:
        return "FLAT/FAIL", acc
    if acc > 0.5 or still_rising:
        return "LEARNS", acc
    return "INCONCLUSIVE", acc

rows = [
    ("transformer (control)", results.get("transformer")),
    ("custom mamba (post-fix)", results.get("mamba")),
    ("mamba-ssm oracle, fast", results.get("mamba_ssm_oracle_fast")),
    ("mamba-ssm oracle, slow", results.get("mamba_ssm_oracle_slow")),
]

print(f"{'run':26s} {'val_acc':>9s}  {'verdict':>12s}   (chance {chance:.3f})")
print("-" * 62)
for name, rec in rows:
    label, acc = classify(rec, chance)
    acc_str = f"{acc:.4f}" if acc is not None else "  n/a"
    print(f"{name:26s} {acc_str:>9s}  {label:>12s}")

cus_label, _ = classify(results.get("mamba"), chance)
fast_label, _ = classify(results.get("mamba_ssm_oracle_fast"), chance)
slow_label, _ = classify(results.get("mamba_ssm_oracle_slow"), chance)
tra_label, _ = classify(results.get("transformer"), chance)

print()
if tra_label != "LEARNS":
    print("INCONCLUSIVE -- even the transformer control did not clearly learn. "
          "Fix the harness before drawing any conclusion about either Mamba.")
elif fast_label == "LEARNS" and slow_label == "LEARNS" and cus_label != "LEARNS":
    print("Official LEARNS on both paths, custom does not -- fusion is exonerated. "
          "A model-level discrepancy remains (init at construction, residual/norm "
          "wrapper, optimizer grouping, or a training-level detail). Next: "
          "module-level forward/backward parity with weight-copying from the "
          "official block into ours, stage by stage (in_proj -> conv+SiLU -> "
          "x_proj -> dt_proj -> scan [already cleared] -> D+gate -> out_proj).")
elif fast_label == "LEARNS" and slow_label != "LEARNS":
    print("Official LEARNS fast, FAILS slow -- the fused execution path itself is "
          "implicated (not just faster, but different at the training-dynamics "
          "level). This is the more surprising and more interesting outcome -- "
          "worth confirming with a second seed before trusting it.")
elif fast_label != "LEARNS" and slow_label != "LEARNS":
    print("Neither official path clearly learns -- reconsider the oracle setup "
          "itself (dimension mismatch, an oracle-side bug) before concluding "
          "anything about the custom implementation.")
else:
    print("Mixed result outside the anticipated cases -- inspect the per-checkpoint "
          "histories in the saved JSON before interpreting.")